# Notebook 05: Baseline Models

---

## Overview

This notebook establishes **baseline performance** using traditional machine learning models on hand-crafted features extracted from chest X-ray images.

**Objectives:**
1. Extract meaningful features from X-ray images (HOG, texture, edge statistics)
2. Train traditional ML models: Logistic Regression, Random Forest, XGBoost
3. Handle multi-label classification with these models
4. Evaluate and compare baseline performance
5. Establish performance benchmarks for deep learning models to beat

**Why Baseline Models?**

Before jumping to complex deep learning, we need to know:
- Can simple models solve this problem?
- What performance level do we need to beat to justify deep learning?
- Which diseases are "easy" vs "hard" to detect?

**Outputs:**
- Extracted feature dataset
- Trained baseline models
- Performance metrics (AUC-ROC, F1, accuracy per disease)
- Comparison report

---

## 💡 Why Traditional ML for Images?

### The Two Paradigms

**Traditional ML (This Notebook):**
```
Raw Image → Hand-Crafted Features → ML Model → Prediction
   (X-ray)   (HOG, edges, texture)  (LogReg/RF)  (Disease)
```
- **WE** design features (edge detection, texture patterns)
- Model learns from OUR features
- Requires domain expertise to create good features
- Fast to train, interpretable, works with small datasets

**Deep Learning (Notebooks 06-07):**
```
Raw Image → Learned Features → Prediction
   (X-ray)  (CNN learns them)  (Disease)
```
- Model learns BOTH features AND classification
- Requires large datasets and computational power
- Often outperforms hand-crafted features

### Analogy to Tabular Data

**Tabular ML:**
- Features already exist: `age`, `income`, `credit_score`
- You might engineer new features: `debt_to_income_ratio = debt / income`
- Then train: LogisticRegression, RandomForest, XGBoost

**Image ML (Traditional):**
- Raw pixels are like individual atoms of data (not meaningful alone)
- We extract "features" from pixels: edge strength, texture smoothness, shape patterns
- Then train the SAME models: LogisticRegression, RandomForest, XGBoost

**Key Insight:**
Once we extract features from images, we're back to **tabular ML**!
- Each image becomes a row
- Each feature becomes a column (HOG_feature_1, HOG_feature_2, ...)
- We use the exact same sklearn models we'd use for any tabular problem

---

## 1. Setup and Configuration

In [ ]:
# Import libraries
import json
import pickle
import sys
import warnings
from pathlib import Path

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Image processing
import cv2
from PIL import Image
from skimage.feature import graycomatrix, graycoprops, hog
from tqdm.auto import tqdm

# ML models
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

# Configuration
warnings.filterwarnings('ignore')
np.random.seed(42)

print("✓ Libraries imported successfully")

In [ ]:
# Define paths
current_path = Path.cwd()

if current_path.name == 'jupyter_notebooks':
    PROJECT_ROOT = current_path.parent
elif (current_path / 'setup.py').exists() or (current_path / 'README.md').exists():
    PROJECT_ROOT = current_path
else:
    PROJECT_ROOT = current_path.parent

DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
MODELS_DIR = PROJECT_ROOT / 'models' / 'saved_models'

# Create directories
MODELS_DIR.mkdir(exist_ok=True, parents=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data: {PROCESSED_DIR}")
print(f"Models directory: {MODELS_DIR}")

In [ ]:
# Configuration for baseline models
CONFIG = {
    # Feature extraction
    'feature_img_size': (128, 128),  # Resize for feature extraction
    'hog_orientations': 9,
    'hog_pixels_per_cell': (8, 8),
    'hog_cells_per_block': (2, 2),
    
    # Sampling (to speed up feature extraction)
    'use_sample': True,  # Set to False to use full dataset
    'sample_size': 5000,  # Images per split (train/val/test)
    
    # Model parameters
    'random_state': 42,
    'n_jobs': -1,  # Use all CPU cores
}

print("Baseline Models Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 2. Load Preprocessed Splits and Configuration

In [ ]:
# Load split files from Notebook 03
train_df = pd.read_csv(PROCESSED_DIR / 'train_split.csv')
val_df = pd.read_csv(PROCESSED_DIR / 'val_split.csv')
test_df = pd.read_csv(PROCESSED_DIR / 'test_split.csv')

print(f"✓ Loaded splits:")
print(f"  Train: {len(train_df):,} images")
print(f"  Val:   {len(val_df):,} images")
print(f"  Test:  {len(test_df):,} images")

# Load preprocessing config
with open(PROCESSED_DIR / 'preprocessing_config.json', 'r') as f:
    prep_config = json.load(f)

disease_classes = prep_config['disease_classes']
print(f"\n✓ Disease classes: {len(disease_classes)}")

In [ ]:
# Optionally sample for faster experimentation
if CONFIG['use_sample']:
    sample_size = CONFIG['sample_size']
    train_df = train_df.sample(n=min(sample_size, len(train_df)), random_state=42)
    val_df = val_df.sample(n=min(sample_size // 5, len(val_df)), random_state=42)
    test_df = test_df.sample(n=min(sample_size // 5, len(test_df)), random_state=42)
    
    print(f"⚠️ Using sample mode for faster experimentation:")
    print(f"  Train: {len(train_df):,} images")
    print(f"  Val:   {len(val_df):,} images")
    print(f"  Test:  {len(test_df):,} images")
    print(f"\n  💡 Set CONFIG['use_sample'] = False to use full dataset")
else:
    print("Using full dataset")

## 3. Feature Extraction Functions

### 📚 What We're Extracting

We extract three types of features from each X-ray image:

#### 1. HOG (Histogram of Oriented Gradients)
- Captures **edge directions** in the image
- Works by dividing image into small cells and computing gradient orientations
- **Medical relevance:** Lungs, ribs, and abnormal masses have characteristic edge patterns
- **Analogy:** Like describing a person by their silhouette shape
- **Output:** ~1,764 features (depends on image size and parameters)

#### 2. Texture Features (GLCM - Gray-Level Co-occurrence Matrix)
- Captures **texture patterns** (smooth, rough, repetitive)
- Measures spatial relationships between pixel intensities
- **Medical relevance:** 
  - Normal lung tissue has fine, homogeneous texture
  - Pneumonia creates cloudy, irregular texture
  - Fibrosis shows reticular (net-like) patterns
- **Properties extracted:**
  - Contrast: Local intensity variation
  - Homogeneity: Texture smoothness
  - Energy: Uniformity of texture
  - Correlation: Linear dependencies in texture
- **Output:** 16 features (4 properties × 4 directions)

#### 3. Statistical Features
- Basic pixel intensity statistics
- **Features:**
  - Mean intensity: Overall brightness
  - Std intensity: Contrast level
  - Edge density: Amount of edges (using Canny edge detection)
- **Medical relevance:**
  - Cardiomegaly: Lower mean intensity (large heart shadow)
  - Pneumothorax: Higher edge density (collapsed lung border)
  - Emphysema: Lower overall intensity (dark, overinflated lungs)
- **Output:** 3 features

### Total Feature Count
~1,783 features per image (HOG + GLCM + Stats)

### Comparison to Tabular ML
- **Tabular data:** Features already exist (age, income, etc.)
- **Image data:** We CREATE features from raw pixels
- Once created, both use the same ML models (LogReg, RF, XGBoost)

---

In [ ]:
def extract_hog_features(image_path, img_size=(128, 128)):
    """
    Extract HOG (Histogram of Oriented Gradients) features.
    
    HOG captures edge patterns and shapes in the image.
    """
    # Load and resize image
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    
    img = cv2.resize(img, img_size)
    
    # Extract HOG features
    features = hog(
        img,
        orientations=CONFIG['hog_orientations'],
        pixels_per_cell=CONFIG['hog_pixels_per_cell'],
        cells_per_block=CONFIG['hog_cells_per_block'],
        visualize=False,
        feature_vector=True
    )
    
    return features


def extract_texture_features(image_path, img_size=(128, 128)):
    """
    Extract texture features using GLCM (Gray-Level Co-occurrence Matrix).
    
    GLCM captures texture patterns (smooth, rough, repetitive).
    """
    # Load and resize
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    
    img = cv2.resize(img, img_size)
    
    # Reduce to 8 gray levels for GLCM (faster computation)
    img = (img // 32).astype(np.uint8)
    
    # Compute GLCM in 4 directions
    distances = [1]
    angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    
    glcm = graycomatrix(
        img, 
        distances=distances, 
        angles=angles,
        levels=8,
        symmetric=True, 
        normed=True
    )
    
    # Extract texture properties
    contrast = graycoprops(glcm, 'contrast').flatten()
    homogeneity = graycoprops(glcm, 'homogeneity').flatten()
    energy = graycoprops(glcm, 'energy').flatten()
    correlation = graycoprops(glcm, 'correlation').flatten()
    
    features = np.concatenate([contrast, homogeneity, energy, correlation])
    
    return features


def extract_statistical_features(image_path, img_size=(128, 128)):
    """
    Extract basic statistical features from pixel intensities.
    """
    # Load and resize
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    
    img = cv2.resize(img, img_size)
    
    # Statistical features
    mean_intensity = np.mean(img)
    std_intensity = np.std(img)
    
    # Edge density using Canny edge detection
    edges = cv2.Canny(img, 50, 150)
    edge_density = np.sum(edges > 0) / edges.size
    
    features = np.array([mean_intensity, std_intensity, edge_density])
    
    return features


def extract_all_features(image_path, img_size=(128, 128)):
    """
    Extract all features from a single image.
    
    Returns:
        Combined feature vector (HOG + Texture + Statistical)
    """
    hog_feat = extract_hog_features(image_path, img_size)
    texture_feat = extract_texture_features(image_path, img_size)
    stat_feat = extract_statistical_features(image_path, img_size)
    
    if hog_feat is None or texture_feat is None or stat_feat is None:
        return None
    
    # Concatenate all features
    all_features = np.concatenate([hog_feat, texture_feat, stat_feat])
    
    return all_features


print("✓ Feature extraction functions defined")
print(f"\n  Feature types:")
print(f"    - HOG (edge patterns)")
print(f"    - GLCM (texture)")
print(f"    - Statistical (intensity, edges)")

## 4. Extract Features from All Images

### 📚 What We're Doing

This is analogous to **feature engineering** in tabular ML.

**In tabular ML:**
```python
# Start with raw features
df['debt_to_income'] = df['debt'] / df['income']
df['age_squared'] = df['age'] ** 2
df['is_weekend'] = df['day_of_week'].isin(['Sat', 'Sun'])

# Now ready for model training
```

**In image ML (this step):**
```python
# Start with image paths
for image_path in images:
    features = extract_hog + extract_texture + extract_stats
    # Convert image → feature vector

# Now ready for model training
```

**Result:**
- Before: DataFrame with `image_path` and disease labels
- After: DataFrame with ~1,783 feature columns and disease labels
- Exactly like tabular data now!

**Time estimate:**
- Sample mode (5,000 images): ~5-10 minutes
- Full dataset (112K images): ~2-3 hours

---

In [ ]:
def extract_features_from_df(df, img_size=(128, 128)):
    """
    Extract features from all images in a dataframe.
    
    Args:
        df: DataFrame with 'full_path' column
        img_size: Target image size for feature extraction
    
    Returns:
        feature_matrix: (n_images, n_features) array
        valid_indices: Indices of successfully processed images
    """
    features_list = []
    valid_indices = []
    
    print(f"Extracting features from {len(df):,} images...")
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        img_path = row['full_path']
        features = extract_all_features(img_path, img_size)
        
        if features is not None:
            features_list.append(features)
            valid_indices.append(idx)
    
    feature_matrix = np.array(features_list)
    
    print(f"✓ Extracted features: {feature_matrix.shape}")
    print(f"  Failed: {len(df) - len(valid_indices)} images")
    
    return feature_matrix, valid_indices


print("✓ Batch feature extraction function defined")

In [ ]:
# Extract features from train set
print("\n" + "="*60)
print("TRAIN SET")
print("="*60)
X_train, train_valid_idx = extract_features_from_df(
    train_df, 
    img_size=CONFIG['feature_img_size']
)

# Update train_df to only valid images
train_df = train_df.loc[train_valid_idx].reset_index(drop=True)
y_train = train_df[disease_classes].values

In [ ]:
# Extract features from validation set
print("\n" + "="*60)
print("VALIDATION SET")
print("="*60)
X_val, val_valid_idx = extract_features_from_df(
    val_df,
    img_size=CONFIG['feature_img_size']
)

val_df = val_df.loc[val_valid_idx].reset_index(drop=True)
y_val = val_df[disease_classes].values

In [ ]:
# Extract features from test set
print("\n" + "="*60)
print("TEST SET")
print("="*60)
X_test, test_valid_idx = extract_features_from_df(
    test_df,
    img_size=CONFIG['feature_img_size']
)

test_df = test_df.loc[test_valid_idx].reset_index(drop=True)
y_test = test_df[disease_classes].values

In [ ]:
# Summary
print("\n" + "="*60)
print("FEATURE EXTRACTION COMPLETE")
print("="*60)
print(f"Train: {X_train.shape[0]:,} images × {X_train.shape[1]:,} features")
print(f"Val:   {X_val.shape[0]:,} images × {X_val.shape[1]:,} features")
print(f"Test:  {X_test.shape[0]:,} images × {X_test.shape[1]:,} features")
print(f"\nLabel shape:")
print(f"Train: {y_train.shape} ({len(disease_classes)} diseases)")
print(f"Val:   {y_val.shape}")
print(f"Test:  {y_test.shape}")

## 5. Feature Scaling

### 📚 What We're Doing

This is **exactly the same** as StandardScaler in tabular ML!

**Why scale features?**

Our extracted features have wildly different ranges:
- Mean intensity: 0-255
- HOG features: 0.0-0.5
- Edge density: 0.0-1.0
- Texture contrast: 0-100

**Problem:**
- Models like Logistic Regression and XGBoost are sensitive to feature scales
- Features with larger magnitudes dominate the learning
- Example: `intensity=200` has more influence than `edge_density=0.5`

**Solution: StandardScaler (z-score normalization)**

```python
scaled_value = (value - mean) / std
```

After scaling, ALL features have:
- Mean = 0
- Standard deviation = 1
- Equal influence on model training

**Critical: Fit only on training data!**
```python
scaler.fit(X_train)  # Learn mean/std from training data
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)    # Apply same scaling to validation
X_test = scaler.transform(X_test)  # Apply same scaling to test
```

**Why?** Prevents data leakage - test set statistics shouldn't influence training!

---

In [ ]:
# Standardize features (fit on train, transform all)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("✓ Features standardized (mean=0, std=1)")
print(f"\nTrain set statistics after scaling:")
print(f"  Mean: {X_train_scaled.mean():.6f}")
print(f"  Std:  {X_train_scaled.std():.6f}")

# Save scaler for deployment
with open(MODELS_DIR / 'baseline_feature_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print(f"\n✓ Saved scaler to {MODELS_DIR / 'baseline_feature_scaler.pkl'}")

## 6. Train Baseline Models

### 📚 Multi-Label Classification Strategy

**The Challenge:**
- Standard sklearn models expect **single-label** classification
- We have **multi-label** classification (14 diseases, multiple can be true)

**Solution: MultiOutputClassifier**

This wrapper trains **one binary classifier per disease**:

```python
MultiOutputClassifier(LogisticRegression())
# Internally creates 14 separate models:
#   Model 1: Atelectasis (Yes/No)
#   Model 2: Cardiomegaly (Yes/No)
#   Model 3: Consolidation (Yes/No)
#   ...
#   Model 14: Pneumothorax (Yes/No)
```

**Analogy to Tabular ML:**

Imagine predicting multiple outcomes for a customer:
- Will buy product A? (Yes/No)
- Will buy product B? (Yes/No)
- Will churn? (Yes/No)

You could:
1. ❌ Train 1 model with 8 classes (combinations) → Doesn't scale
2. ✅ Train 3 independent binary models → **MultiOutputClassifier**

---

### Models We're Training

#### 1. Logistic Regression
- Simplest linear classifier
- Fast training, interpretable
- Works well as baseline
- Uses `class_weight='balanced'` to handle class imbalance

#### 2. Random Forest
- Ensemble of decision trees
- Handles non-linear patterns
- More complex than LogReg
- Uses `class_weight='balanced'`

#### 3. XGBoost
- Gradient boosting (state-of-art for tabular data)
- Often wins Kaggle competitions
- Uses `scale_pos_weight` for imbalance

All three are **exactly the models you'd use for tabular ML** - we just extracted features first!

---

### Model 1: Logistic Regression

In [ ]:
print("\n" + "="*60)
print("TRAINING: Logistic Regression")
print("="*60)

# Create multi-output logistic regression
lr_model = MultiOutputClassifier(
    LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=CONFIG['random_state'],
        n_jobs=CONFIG['n_jobs']
    )
)

# Train
lr_model.fit(X_train_scaled, y_train)

# Predictions
lr_train_pred = lr_model.predict(X_train_scaled)
lr_val_pred = lr_model.predict(X_val_scaled)
lr_test_pred = lr_model.predict(X_test_scaled)

# Predict probabilities for AUC-ROC
lr_val_proba = np.array([clf.predict_proba(X_val_scaled)[:, 1] 
                         for clf in lr_model.estimators_]).T

print("✓ Training complete")

# Save model
with open(MODELS_DIR / 'baseline_logistic_regression.pkl', 'wb') as f:
    pickle.dump(lr_model, f)
print(f"✓ Saved model to {MODELS_DIR / 'baseline_logistic_regression.pkl'}")

### Model 2: Random Forest

In [ ]:
print("\n" + "="*60)
print("TRAINING: Random Forest")
print("="*60)

# Create multi-output random forest
rf_model = MultiOutputClassifier(
    RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        class_weight='balanced',
        random_state=CONFIG['random_state'],
        n_jobs=CONFIG['n_jobs']
    )
)

# Train
rf_model.fit(X_train_scaled, y_train)

# Predictions
rf_train_pred = rf_model.predict(X_train_scaled)
rf_val_pred = rf_model.predict(X_val_scaled)
rf_test_pred = rf_model.predict(X_test_scaled)

# Predict probabilities
rf_val_proba = np.array([clf.predict_proba(X_val_scaled)[:, 1] 
                         for clf in rf_model.estimators_]).T

print("✓ Training complete")

# Save model
with open(MODELS_DIR / 'baseline_random_forest.pkl', 'wb') as f:
    pickle.dump(rf_model, f)
print(f"✓ Saved model to {MODELS_DIR / 'baseline_random_forest.pkl'}")

### Model 3: XGBoost

In [ ]:
print("\n" + "="*60)
print("TRAINING: XGBoost")
print("="*60)

# Create multi-output XGBoost
xgb_model = MultiOutputClassifier(
    XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=CONFIG['random_state'],
        n_jobs=CONFIG['n_jobs'],
        eval_metric='logloss'
    )
)

# Train
xgb_model.fit(X_train_scaled, y_train)

# Predictions
xgb_train_pred = xgb_model.predict(X_train_scaled)
xgb_val_pred = xgb_model.predict(X_val_scaled)
xgb_test_pred = xgb_model.predict(X_test_scaled)

# Predict probabilities
xgb_val_proba = np.array([clf.predict_proba(X_val_scaled)[:, 1] 
                          for clf in xgb_model.estimators_]).T

print("✓ Training complete")

# Save model
with open(MODELS_DIR / 'baseline_xgboost.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)
print(f"✓ Saved model to {MODELS_DIR / 'baseline_xgboost.pkl'}")

## 7. Evaluation Functions

In [ ]:
def evaluate_multi_label_model(y_true, y_pred, y_proba, disease_names):
    """
    Evaluate multi-label classification performance.
    
    Args:
        y_true: True labels (n_samples, n_diseases)
        y_pred: Predicted labels (n_samples, n_diseases)
        y_proba: Predicted probabilities (n_samples, n_diseases)
        disease_names: List of disease names
    
    Returns:
        Dictionary of metrics per disease
    """
    results = {}
    
    for i, disease in enumerate(disease_names):
        y_true_disease = y_true[:, i]
        y_pred_disease = y_pred[:, i]
        y_proba_disease = y_proba[:, i]
        
        # Calculate metrics
        accuracy = accuracy_score(y_true_disease, y_pred_disease)
        precision = precision_score(y_true_disease, y_pred_disease, zero_division=0)
        recall = recall_score(y_true_disease, y_pred_disease, zero_division=0)
        f1 = f1_score(y_true_disease, y_pred_disease, zero_division=0)
        
        # AUC-ROC (only if both classes present)
        if len(np.unique(y_true_disease)) > 1:
            auc = roc_auc_score(y_true_disease, y_proba_disease)
        else:
            auc = np.nan
        
        results[disease] = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'auc': auc
        }
    
    return results


print("✓ Evaluation function defined")

## 8. Compare Model Performance

In [ ]:
# Evaluate all models on validation set
print("\n" + "="*60)
print("VALIDATION SET PERFORMANCE")
print("="*60)

lr_results = evaluate_multi_label_model(y_val, lr_val_pred, lr_val_proba, disease_classes)
rf_results = evaluate_multi_label_model(y_val, rf_val_pred, rf_val_proba, disease_classes)
xgb_results = evaluate_multi_label_model(y_val, xgb_val_pred, xgb_val_proba, disease_classes)

print("\n✓ Model evaluation complete")

In [ ]:
# Create comparison dataframe
comparison_data = []

for disease in disease_classes:
    comparison_data.append({
        'Disease': disease,
        'LR_AUC': lr_results[disease]['auc'],
        'LR_F1': lr_results[disease]['f1'],
        'RF_AUC': rf_results[disease]['auc'],
        'RF_F1': rf_results[disease]['f1'],
        'XGB_AUC': xgb_results[disease]['auc'],
        'XGB_F1': xgb_results[disease]['f1'],
    })

comparison_df = pd.DataFrame(comparison_data)

# Display comparison
print("\n" + "="*60)
print("MODEL COMPARISON (Validation Set)")
print("="*60)
print(comparison_df.to_string(index=False))

# Calculate average performance
print("\n" + "-"*60)
print("AVERAGE PERFORMANCE")
print("-"*60)
print(f"Logistic Regression - AUC: {comparison_df['LR_AUC'].mean():.3f}, F1: {comparison_df['LR_F1'].mean():.3f}")
print(f"Random Forest       - AUC: {comparison_df['RF_AUC'].mean():.3f}, F1: {comparison_df['RF_F1'].mean():.3f}")
print(f"XGBoost             - AUC: {comparison_df['XGB_AUC'].mean():.3f}, F1: {comparison_df['XGB_F1'].mean():.3f}")

## 9. Visualize Performance Comparison

In [ ]:
# Plot AUC comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

x = np.arange(len(disease_classes))
width = 0.25

# AUC comparison
ax1.bar(x - width, comparison_df['LR_AUC'], width, label='Logistic Regression', alpha=0.8)
ax1.bar(x, comparison_df['RF_AUC'], width, label='Random Forest', alpha=0.8)
ax1.bar(x + width, comparison_df['XGB_AUC'], width, label='XGBoost', alpha=0.8)

ax1.set_xlabel('Disease', fontweight='bold')
ax1.set_ylabel('AUC-ROC', fontweight='bold')
ax1.set_title('AUC-ROC Comparison Across Baseline Models', fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(disease_classes, rotation=45, ha='right')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim([0, 1])

# F1 comparison
ax2.bar(x - width, comparison_df['LR_F1'], width, label='Logistic Regression', alpha=0.8)
ax2.bar(x, comparison_df['RF_F1'], width, label='Random Forest', alpha=0.8)
ax2.bar(x + width, comparison_df['XGB_F1'], width, label='XGBoost', alpha=0.8)

ax2.set_xlabel('Disease', fontweight='bold')
ax2.set_ylabel('F1 Score', fontweight='bold')
ax2.set_title('F1 Score Comparison Across Baseline Models', fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(disease_classes, rotation=45, ha='right')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)
ax2.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_baseline_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved comparison plot to outputs/figures/")

## 10. Test Set Evaluation (Best Model)

In [ ]:
# Determine best model based on average validation AUC
avg_aucs = {
    'Logistic Regression': comparison_df['LR_AUC'].mean(),
    'Random Forest': comparison_df['RF_AUC'].mean(),
    'XGBoost': comparison_df['XGB_AUC'].mean()
}

best_model_name = max(avg_aucs, key=avg_aucs.get)
print(f"\n🏆 Best model (by avg AUC): {best_model_name}")
print(f"   Average AUC: {avg_aucs[best_model_name]:.3f}")

# Select best model predictions
if best_model_name == 'Logistic Regression':
    best_pred = lr_test_pred
    best_proba = np.array([clf.predict_proba(X_test_scaled)[:, 1] 
                           for clf in lr_model.estimators_]).T
elif best_model_name == 'Random Forest':
    best_pred = rf_test_pred
    best_proba = np.array([clf.predict_proba(X_test_scaled)[:, 1] 
                           for clf in rf_model.estimators_]).T
else:
    best_pred = xgb_test_pred
    best_proba = np.array([clf.predict_proba(X_test_scaled)[:, 1] 
                           for clf in xgb_model.estimators_]).T

# Evaluate on test set
test_results = evaluate_multi_label_model(
    y_test, best_pred, best_proba, disease_classes
)

print("\n" + "="*60)
print(f"TEST SET PERFORMANCE - {best_model_name}")
print("="*60)
print(f"{'Disease':<20} {'AUC':>8} {'F1':>8} {'Precision':>10} {'Recall':>8}")
print("-"*60)

for disease in disease_classes:
    metrics = test_results[disease]
    print(f"{disease:<20} {metrics['auc']:>8.3f} {metrics['f1']:>8.3f} "
          f"{metrics['precision']:>10.3f} {metrics['recall']:>8.3f}")

print("-"*60)
avg_auc = np.mean([test_results[d]['auc'] for d in disease_classes if not np.isnan(test_results[d]['auc'])])
avg_f1 = np.mean([test_results[d]['f1'] for d in disease_classes])
print(f"{'AVERAGE':<20} {avg_auc:>8.3f} {avg_f1:>8.3f}")

## 11. Save Results and Summary

In [ ]:
# Save all results
baseline_results = {
    'config': CONFIG,
    'feature_count': X_train.shape[1],
    'models': {
        'logistic_regression': {
            'val_avg_auc': comparison_df['LR_AUC'].mean(),
            'val_avg_f1': comparison_df['LR_F1'].mean(),
            'per_disease': lr_results
        },
        'random_forest': {
            'val_avg_auc': comparison_df['RF_AUC'].mean(),
            'val_avg_f1': comparison_df['RF_F1'].mean(),
            'per_disease': rf_results
        },
        'xgboost': {
            'val_avg_auc': comparison_df['XGB_AUC'].mean(),
            'val_avg_f1': comparison_df['XGB_F1'].mean(),
            'per_disease': xgb_results
        }
    },
    'best_model': {
        'name': best_model_name,
        'test_avg_auc': avg_auc,
        'test_avg_f1': avg_f1,
        'per_disease': test_results
    }
}

# Convert numpy types to Python native types for JSON serialization
def convert_to_serializable(obj):
    if isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (np.integer, np.int64)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, tuple):
        return tuple(convert_to_serializable(item) for item in obj)
    else:
        return obj

baseline_results = convert_to_serializable(baseline_results)

# Save to JSON
with open(OUTPUTS_DIR / 'reports' / '05_baseline_models_results.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)

print(f"✓ Saved results to {OUTPUTS_DIR / 'reports' / '05_baseline_models_results.json'}")

## 12. Summary and Next Steps

In [ ]:
print("="*60)
print("  ✅ Notebook 05 Complete: Baseline Models")
print("="*60)

print("\n📊 Dataset:")
print(f"  Train: {X_train.shape[0]:,} images")
print(f"  Val:   {X_val.shape[0]:,} images")
print(f"  Test:  {X_test.shape[0]:,} images")

print("\n🔧 Features Extracted:")
print(f"  Total features per image: {X_train.shape[1]:,}")
print(f"    - HOG features (edge patterns)")
print(f"    - GLCM features (texture)")
print(f"    - Statistical features")

print("\n🤖 Models Trained:")
print(f"  1. Logistic Regression - Val AUC: {comparison_df['LR_AUC'].mean():.3f}")
print(f"  2. Random Forest       - Val AUC: {comparison_df['RF_AUC'].mean():.3f}")
print(f"  3. XGBoost             - Val AUC: {comparison_df['XGB_AUC'].mean():.3f}")

print(f"\n🏆 Best Model: {best_model_name}")
print(f"  Test AUC: {avg_auc:.3f}")
print(f"  Test F1:  {avg_f1:.3f}")

print("\n📁 Generated Files:")
print(f"  {MODELS_DIR / 'baseline_logistic_regression.pkl'}")
print(f"  {MODELS_DIR / 'baseline_random_forest.pkl'}")
print(f"  {MODELS_DIR / 'baseline_xgboost.pkl'}")
print(f"  {MODELS_DIR / 'baseline_feature_scaler.pkl'}")
print(f"  {OUTPUTS_DIR / 'reports' / '05_baseline_models_results.json'}")

print("\n💡 Key Insights:")
print("  - Traditional ML can detect some diseases from hand-crafted features")
print("  - Performance varies significantly across diseases")
print("  - These results establish baseline for deep learning to beat")

print("\n⏭️  Ready for Notebook 06: Deep Learning Models!")
print("  Deep CNNs will learn features automatically from raw pixels")